In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://karnatakabank.com/deposit-interest-rates" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find_all("table")
    table2=table[1]
    rows=table2.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[1:]:  #Skip heading
        data = i.find_all("td")
        row=['Karnataka Bank']
        row.extend([tr.text for tr in data[:2]])
        for i in range(0,3):
            row[i] = ' '.join(str(row[i]).strip().replace('\n', '').split())
        if row_count < 5: row.extend(['',str((float(row[2][0:4]) + 0.25)) + '%',''])
        else: row.extend(['',str((float(row[2][0:4]) + 0.50)) + '%',''])
        row[2] = row[2] + '%'
        print("row=",row)
        if row_count == 6: 
            tax_saver_row = ['Karnataka Bank', 'Tax-saver Fixed Deposit', row[2], '', row[4],'',]
            list_of_interest_rates.append(tax_saver_row)
        elif row_count == 5:
            row[1] = '375 days'
        list_of_interest_rates.append(row)
        row_count += 1
    
    #print(list_of_interest_rates)
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nKarnataka Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 09122024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_09122024.csv
row= ['Karnataka Bank', '7 days to 45 days', '3.50%', '', '3.75%', '']
row= ['Karnataka Bank', '46 days to 90 days', '4.00%', '', '4.25%', '']
row= ['Karnataka Bank', '91 days to 179 days', '5.25%', '', '5.5%', '']
row= ['Karnataka Bank', '180 days to below 1 year', '6.25%', '', '6.5%', '']
row= ['Karnataka Bank', '1 year to 2 years', '7.25%', '', '7.5%', '']
row= ['Karnataka Bank', '375 days (FD & ACC)', '7.50%', '', '8.0%', '']
row= ['Karnataka Bank', 'Above 2 years to 5 years', '6.50%', '', '7.0%', '']
row= ['Karnataka Bank', 'Above 5 years to 10 years', '5.80%', '', '6.3%', '']
        bank_name                     tenure gen_rate annualised_gen_rate  \
0  Karnataka Bank          7 days to 45 days    3.50%                       
1  Karnataka Bank         46 days to 90 days    4.00%                       
2  Karnataka Bank        91 days to 179 days    5.25%                       
3 